<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/NeuroLearn/blob/main/SlideMDCharClassifier_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SlideMD Subword Classifier
### Goal
The goal of this neural network is to take a sentence and to identify what subwords in the sentence would make for good terms to be clozed in a cloze card.

### How it will work
We will do this using Karpathy's CNNs, but modified. Essentially, we will have a sliding window that goes over all the words. It will essentially put one subword in the middle, and it will look at all the subword to come and all the subwords that have gone. Based on that, it will determine if it a good subword to cloze or not. We will cloze words by averaging the subword's clozability score in each word. We will only cloze one word per sentence, which will be the word with the highest clozability score.  

On top of that, we will use a sub-word encryption instead of a character, because our dataset is tweeny, and it will be faster for the model to learn subwords then characters.

In [ ]:
import numpy as np
import numpy.lib.stride_tricks as lst
import pandas as pd
import math
import matplotlib.pyplot as plt
from google.colab import drive
import re
import itertools
import random
%matplotlib inline

In [ ]:
# Hyperparameters
MAX_WORD_LEN = 35 # Max Padding Length Characters in a Word
MAX_SENT_LEN = 200 # Max Padding Length Words in a Sentence
BATCH_SIZE = 16 # Batch Size
N_EMBD = 100 # 100 Embedding Dimensions
WIN_SIZE = 5 # Sliding Window Size
NEURONS_L1 = 256 # Number of Neurons Layer 1
NEURONS_L2 = 64 # Number of Neurons Layer 2
EPOCHS = 16

In [ ]:
# Grabbing the dataset
file_path = ''
try:
  drive.mount('/content/gdrive')
  file_path = '/content/gdrive/MyDrive/SlideMD/proofOfConceptDatasetSMSCC.txt'
except Exception as e:
  file_path = 'proofOfConceptDatasetSMSCC.txt'

# Load the dataset
df = pd.read_json(file_path, lines = True)
df = df[['question', 'answer']]

Mounted at /content/gdrive


In [ ]:
# Cleaning the Dataset
# First Lower-casing the entire dataset
df['question'] = df['question'].str.lower()
df['answer'] = df['answer'].str.lower()

# Removing all non-letter characters
df['question'] = df['question'].str.replace(r'[^a-z\s_]', '', regex=True)
df['answer'] = df['answer'].str.replace(r'[^a-z\s]', '', regex=True)
df['question'] = df['question'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Replacing the underscores with
questions = df['question'].tolist()
answers = df['answer'].tolist()

temp_combined = list(zip(questions, answers))
# print(questions[:2], answers[:2])
random.shuffle(temp_combined)
questions, answers = map(list, zip(*temp_combined))
# print(questions[:2], answers[:2])

# Removing all bad data points
i = 0
while i < len(questions):
  underscores = re.findall(r'\b_\w+', questions[i])
  if len(underscores) != 1:
    bq = questions.pop(i)
    ba = answers.pop(i)
    # print(bq, ba)
  else:
    i += 1

# Creating the sentences and index of the words that we need
sentences = []
indexes = []
for i, question in enumerate(questions):
  underscores = re.findall(r'\b_\w+', question)
  if len(underscores) != 1:
    print(f"ERROR 1: Expected 1 underscore found {len(underscores)} at index {i}")
  index = (question.split()).index(underscores[0])
  indexes.append(index)
  question = question.replace(underscores[0], answers[i])
  sentences.append(question)

# for question, answer in list(zip(questions, answers))[:8]:
#   print(question, " : " , answer)

# print("-----"*40)
# for sentence, index in list(zip(sentences, indexes))[:8]:
#   print(sentence, " : ", index)

In [ ]:
# Creating the Indexing Dictionaries
stoi = {character: index + 2 for index, character in enumerate(list(map(chr, range(ord('a'), ord('z') + 1))))}
stoi['*'] = 0 # Padding Character
stoi['.'] = 1 # Unknown Character
itos = {index: character for character, index in stoi.items()}

In [ ]:
# Create a word to number tokenizer
def tokenize_char(word):
  ans = [stoi[char] for char in word]
  if len(ans) < MAX_WORD_LEN:
    ans += [0] * (MAX_WORD_LEN - len(ans))
  else:
    print(f"ERROR 2: the word, {word}, exceeds the mpl by {MAX_WORD_LEN-len(ans)}")
  return ans

In [ ]:
# Transforming the numbers into Strings
enc_sent = [] # Transformed Sentences
for sentence in sentences:
  words = sentence.split()
  enc_senti = list(map(tokenize_char, words))

  # Adding Padding Words for the rest of the sentence
  if len(enc_senti) < MAX_SENT_LEN:
    padding = [0] * MAX_WORD_LEN
    enc_senti.extend(padding for i in range(MAX_SENT_LEN - len(enc_senti)))
  else:
    print(f"ERROR 3: The sentence length exceeded mpls by {len(enc_senti) - MAX_SENT_LEN}")
  enc_sent.append(enc_senti)

In [ ]:
# Separating the Numpy Sentences and Indexes
raw_inputs = np.array(enc_sent, dtype=np.float32) # Sentences In Numbers & Padded
xshape = raw_inputs.shape
Y = np.array(indexes, dtype=np.int32) # Indexes of Answer Word
xshape

(1695, 200, 35)

We need to randomly select 20% of the indicies and make sure the model does NOT have their subwords memorized. This is to ensure that the model knows how to deal with the UNK token.

We must also create a training split and test split.

In [ ]:
## Creating the Subword Embeddings
selected_tokens = np.random.choice(xshape[0], size = int(0.8 * xshape[0]), replace=False)
words = raw_inputs[selected_tokens].reshape(-1, MAX_WORD_LEN)  # All Vocab Words
grams = lst.sliding_window_view(words, window_shape=4, axis = 1).reshape(-1, 4) # All Four Grams

# Removing All Non-Unique 4 Grams
gramvoid = np.dtype((np.void, grams.dtype.itemsize * 4)) # Creating a Void Structure of Gram Size
grams = np.ascontiguousarray(grams).view(gramvoid).ravel() # Turning this into 1D Array to use Unique

# Inserting Unknown Character
vocab = np.unique(grams, axis = 0) # Unique Four Grams
unk = np.ascontiguousarray([1]*4).view(gramvoid) # Viewing the 2D array as an obj
unk_idx = np.searchsorted(vocab, unk) # Finding where to add
vocab = np.insert(vocab, unk_idx, unk) # Adding the New Unknown 4 Gram

# Turning Dtype back to Normal
vocab = vocab.view(words.dtype)

grams.shape, vocab.shape, unk_idx

((8678400,), (46716,), array([11677, 11677]))

## Future Addition
Instead of treating the size-4 window as a rigid rule, their vocabularies contain a mix of whole subwords and individual characters.

If the tokenizer encounters a word like "beetle", it first checks if "beetle" is in the vocabulary.

If it isn't, it breaks it down into smaller pieces, like "beet" and "le".

If "beet" is missing, it breaks it down further, eventually falling back to individual characters ("b", "e", "e", "t").

Because every single individual letter of the alphabet is guaranteed to be in the vocabulary, the model can always construct a representation for any word, meaning it never encounters a true "unknown."

In [ ]:
# Creating my own custom search sorted that does return 1
def lookup(arr, val, unk_idx): # Strict Search Sorted
  if len(arr) == 0:
    return np.ones_like(val) * unk_idx[0]

  # Getting where d ints are
  idx = np.searchsorted(arr, val)
  cidx = np.minimum(idx, len(arr) - 1)

  return np.where(((idx < len(arr)) & (arr[cidx] == val)), idx, unk_idx[0])

In [ ]:
# Using a Lookup table to replace inputs
grams = grams.view(vocab.dtype) # Getting Every Single 4 Gram

# Creating A New Void Array Datatype for fast comparison
words = raw_inputs.reshape(-1, MAX_WORD_LEN)  # All Vocab Words
grams = lst.sliding_window_view(words, window_shape=4, axis = 1).reshape(-1, 4) # All Four Grams
X_void = np.ascontiguousarray(grams).view(gramvoid).ravel()
vocab_void = np.ascontiguousarray(vocab).view(gramvoid).ravel()

# Replacing The New Inputs with the Indexes
indices = lookup(vocab_void, X_void, unk_idx)

# Reshape back to (Sentences, Words, Windows_per_word)
X = indices.reshape(xshape[0], xshape[1], -1)
Xtr, Xte = X[:int(0.8 * xshape[0])], X[int(0.8 * xshape[0]):]
Ytr, Yte = Y[:int(0.8 * xshape[0])], Y[int(0.8 * xshape[0]):]

In [ ]:
# Weights and Biases
C = np.random.randn(vocab.shape[0], N_EMBD).astype(np.float32)
W1 = np.random.randn(N_EMBD * Xtr.shape[2] * WIN_SIZE, NEURONS_L1).astype(np.float32) * np.sqrt(2/(N_EMBD * Xtr.shape[2] * WIN_SIZE))
B1 = np.random.randn(NEURONS_L1).astype(np.float32)
g1 = np.ones((1, 1, NEURONS_L1)).astype(np.float32)
b1 = np.zeros((1, 1, NEURONS_L1)).astype(np.float32)
W2 = np.random.randn(NEURONS_L1 * WIN_SIZE, NEURONS_L2).astype(np.float32) * np.sqrt(2/(NEURONS_L1 * WIN_SIZE))
B2 = np.random.randn(NEURONS_L2).astype(np.float32)
g2 = np.ones((1, 1, NEURONS_L2)).astype(np.float32)
b2 = np.zeros_like(g2).astype(np.float32)
W3 = np.random.randn(NEURONS_L2, 1).astype(np.float32) * 0.01
B3 = np.random.randn(1).astype(np.float32) * 0.01

parameters = [C, W1, B1, g1, b1, W2, B2, g2, b2, W3, B3]

In [ ]:
# Stats
lossi = []
ud = []
steps_per_epoch = -(Xtr.shape[0] // -BATCH_SIZE)
total_steps = EPOCHS * steps_per_epoch
steppi = list(range(total_steps))
step = 0

print(f"Total iterations expected: {total_steps}")

Total iterations expected: 1360


In [ ]:
# Training Loop
for epoch in range(EPOCHS):
  for i in range(-(Xtr.shape[0] // -BATCH_SIZE)):
    ### Forward Pass
    ## Batching
    xbatch = Xtr[list(range(BATCH_SIZE * i, min(BATCH_SIZE * i + BATCH_SIZE - 1, Xtr.shape[0])))]
    ybatch = Ytr[list(range(BATCH_SIZE * i, min(BATCH_SIZE * i + BATCH_SIZE - 1, Xtr.shape[0])))]

    ## Layer 0
    # Embedding the subword tokens
    emb = C[xbatch]
    embcat = emb.reshape(xbatch.shape[0], xbatch.shape[1], -1)
    embcat_shape = embcat.shape
    embcat_dtype = embcat.dtype

    ## Layer 1
    # Padding the sentences
    padding_word = np.tile(C[0], 32)
    pad_block = np.tile(padding_word, (embcat.shape[0], 2, 1))
    pembcat = np.concatenate([pad_block, embcat, pad_block], axis=1)

    # Applying the Slider
    win1 = lst.sliding_window_view(pembcat, window_shape=WIN_SIZE, axis=1)
    win1r = win1.reshape(win1.shape[:-2] + (-1,))
    win1r = np.ascontiguousarray(win1r) # Faster for Matrix Mult

    # First layer output
    h1 = win1r @ W1 + B1

    # LayerNorm
    h1_mean = h1.mean(axis = 2, keepdims=True)
    h1_var = h1.var(axis = 2, keepdims=True)
    preln1 = (h1 - h1_mean)/((h1_var + 1e-5) ** 0.5)
    ln1 = g1 * preln1 + b1

    # ReLu 1
    rl1 = np.maximum(np.zeros_like(ln1), ln1)

    ## Layer 2
    # Padding the Array
    rl1_padded = np.pad(rl1, ((0,0), (2,2), (0,0)), mode='constant', constant_values=0)

    # Applying the Slider
    win2 = lst.sliding_window_view(rl1_padded, window_shape = WIN_SIZE, axis = 1)
    win2r = win2.reshape(win2.shape[:-2] + (-1,))
    win2r = np.ascontiguousarray(win2r)

    # Matrix Multiplying with Weights
    h2 = win2r @ W2 + B2

    # LayerNorm again
    h2_mean = h2.mean(axis = 2, keepdims=True)
    h2_var = h2.var(axis = 2, keepdims=True)
    preln2 = (h2 - h2_mean)/((h2_var + 1e-5) ** 0.5)
    ln2 = g2 * preln2 + b2

    # ReLu
    rl2 = np.maximum(np.zeros_like(ln2), ln2)

    ## Layer 3
    # Final Linear Layer
    h3 = rl2 @ W3 + B3
    h3 = h3.squeeze(axis=-1)

    # Softmax
    logmax = np.max(h3, keepdims=True, axis=1)
    logits = h3 - logmax
    raw_probs = np.e ** logits
    logits_sum = np.sum(raw_probs, axis = 1, keepdims=True)
    probs = raw_probs * logits_sum ** -1

    # Sparse Categorical Crossentropy
    current_batch_size = xbatch.shape[0]
    loss = -np.log(probs[np.arange(current_batch_size), ybatch])

    ### Backward Pass
    # Sparse Categorical Crossentropy
    dloss = 1/current_batch_size

    ## Loss Calculations
    # Softmax
    dprobs = np.zeros_like(probs)
    dprobs[np.arange(current_batch_size), ybatch] = (-1.0 / probs[np.arange(current_batch_size), ybatch]) * dloss
    dlogits_sum = (-1 * (logits_sum ** -2) * raw_probs * dprobs).sum(1, keepdims=True)
    draw_probs = (dprobs * (logits_sum ** -1) + dlogits_sum * (np.ones_like(logits_sum)))
    dlogits = (np.e ** logits) * draw_probs
    dlogmax = (-dlogits * (np.ones_like(h3))).sum(1, keepdims=True)
    dh3 = (dlogits * np.ones_like(logmax)) + (dlogmax * (h3 == logmax))

    ## Layer 3
    # Linear Layer 3
    drl2 = np.expand_dims(dh3, -1) @ W3.T
    dW3 = rl2.reshape(rl2.shape[0] * rl2.shape[1], rl2.shape[-1]).T @ dh3.reshape(rl2.shape[0] * rl2.shape[1], -1) #PARAPARAMETER!!!
    dB3 = (np.ones_like(B3) * dh3).ravel().sum(0, keepdims=True) #PARAPARAMETER!!!

    ## Layer 2
    # ReLu Layer 2
    dln2 = drl2 * (ln2 >= 0)

    # LayerNorm 2
    dg2 = np.sum(preln2 * dln2, axis = (0,1), keepdims=True) #PARAPARAMETER!!!
    db2 = np.sum(np.ones_like(b2) * dln2, axis = (0,1), keepdims=True) #PARAPARAMETER!!!
    dpreln2 = g2 * dln2
    dh2 = (dpreln2 - dpreln2.mean(axis=2, keepdims=True) - preln2 * (dpreln2 * preln2).mean(axis=2, keepdims=True))/(np.sqrt(h2_var + 1e-5))

    # Linear Layer 2
    dW2 = (win2r.reshape(-1, win2r.shape[-1])).T @ dh2.reshape(-1, dh2.shape[-1]) #PARAPARAMETER!!!
    dB2 = dh2.sum((0,1)) #PARAPARAMETER!!!
    dwin2r = (dh2.reshape(-1, dh2.shape[-1]) @ W2.T).reshape(win2r.shape)

    # Convolution 2 + Padding 2
    dwin2 = dwin2r.reshape(win2.shape)
    drl1_padded = np.zeros_like(rl1_padded)
    for w in range(WIN_SIZE):
      drl1_padded[:, w : MAX_SENT_LEN + w, :] += dwin2[:, :, :, w]
    drl1 = drl1_padded[:, 2:-2, :]

    ## Layer 1
    # ReLu Layer 1
    dln1 = drl1 * (ln1 >= 0)

    # LayerNorm 1
    dg1 = np.sum(preln1 * dln1, axis = (0,1), keepdims=True) #PARAPARAMETER!!!
    db1 = np.sum(np.ones_like(b1) * dln1, axis = (0,1), keepdims=True) #PARAPARAMETER!!!
    dpreln1 = g1 * dln1
    dh1 = (dpreln1 - dpreln1.mean(axis=2, keepdims=True) - preln1 * (dpreln1 * preln1).mean(axis=2, keepdims=True))/(np.sqrt(h1_var + 1e-5))

    # Linear Layer 1
    dW1 = (win1r.reshape(-1, win1r.shape[-1])).T @ dh1.reshape(-1, dh1.shape[-1]) #PARAPARAMETER!!!
    dB1 = dh1.sum((0,1)) #PARAPARAMETER!!!
    dwin1r = (dh1.reshape(-1, dh1.shape[-1]) @ W1.T).reshape(win1r.shape)

    # Convolution 1 + Padding 1
    dwin1 = dwin1r.reshape(win1.shape)
    dpembcat = np.zeros_like(pembcat)
    for w in range(WIN_SIZE):
      dpembcat[:, w : MAX_SENT_LEN + w, :] += dwin1[:, :, :, w]
    dembcat = dpembcat[:, 2:-2, :]

    ## Embedding Layer
    demb = dembcat.reshape(emb.shape)
    dC = np.zeros_like(C)
    ix = xbatch.ravel()
    np.add.at(dC, ix, demb.reshape(-1, demb.shape[-1]))

    gradients = [dC, dW1, dB1, dg1, db1, dW2, dB2, dg2, db2, dW3, dB3]

    ### Update Parameters
    lr = 0.01 if epoch < 11 else 0.001
    # lr = lri[step]
    ud_step = []
    for parameter, gradient in list(zip(parameters, gradients)):
      update = lr * gradient
      parameter -= update

      # Finding Ratio
      if i > 0:
        ratio = update.std() / (parameter.std() + 1e-5)
        ud_step.append(np.log10(ratio + 1e-5))
    ud.append(ud_step)

    ### Resetting Gradients
    if epoch == (EPOCHS - 1) and i == (-(Xtr.shape[0] // -BATCH_SIZE) - 1):
      pass
    else:
      for gradient in gradients:
        gradient[:] = 0

    ### Tracking Stats
    if epoch == 0 and i == 0:
      print(f"Initial Loss: {loss.mean()}")
    lossi.append(loss.mean())
    step += 1


  ### Printing Epochs and Loss
  print(f"Epoch {epoch}: Loss {loss.mean()}")
print(step)

Initial Loss: 5.2933548408790205
Epoch 0: Loss 2.762116531060128
Epoch 1: Loss 2.817765027860261
Epoch 2: Loss 2.6205102656912076
Epoch 3: Loss 2.5194334991770346
Epoch 4: Loss 1.7865002097604892
Epoch 5: Loss 1.6555137810500895
Epoch 6: Loss 1.5391681995022273
Epoch 7: Loss 1.289623160067747
Epoch 8: Loss 1.2438114980567037
Epoch 9: Loss 0.9740090656123912
Epoch 10: Loss 0.7887375400870055
Epoch 11: Loss 0.3652740825587138
Epoch 12: Loss 0.32435163915881116
Epoch 13: Loss 0.3254078294109425
Epoch 14: Loss 0.2969444155509494
Epoch 15: Loss 0.2721127947460879
1360


In [ ]:
### Testing Loss Validation
## Batching
xbatch = Xte
ybatch = Yte

## Layer 0
# Embedding the subword tokens
emb = C[xbatch]
embcat = emb.reshape(xbatch.shape[0], xbatch.shape[1], -1)
embcat_shape = embcat.shape
embcat_dtype = embcat.dtype

## Layer 1
# Padding the sentences
padding_word = np.tile(C[0], 32)
pad_block = np.tile(padding_word, (embcat.shape[0], 2, 1))
pembcat = np.concatenate([pad_block, embcat, pad_block], axis=1)

# Applying the Slider
win1 = lst.sliding_window_view(pembcat, window_shape=WIN_SIZE, axis=1)
win1r = win1.reshape(win1.shape[:-2] + (-1,))
win1r = np.ascontiguousarray(win1r) # Faster for Matrix Mult

# First layer output
h1 = win1r @ W1 + B1

# LayerNorm
h1_mean = h1.mean(axis = 2, keepdims=True)
h1_var = h1.var(axis = 2, keepdims=True)
preln1 = (h1 - h1_mean)/((h1_var + 1e-5) ** 0.5)
ln1 = g1 * preln1 + b1

# ReLu 1
rl1 = np.maximum(np.zeros_like(ln1), ln1)

## Layer 2
# Padding the Array
rl1_padded = np.pad(rl1, ((0,0), (2,2), (0,0)), mode='constant', constant_values=0)

# Applying the Slider
win2 = lst.sliding_window_view(rl1_padded, window_shape = WIN_SIZE, axis = 1)
win2r = win2.reshape(win2.shape[:-2] + (-1,))
win2r = np.ascontiguousarray(win2r)

# Matrix Multiplying with Weights
h2 = win2r @ W2 + B2

# LayerNorm again
h2_mean = h2.mean(axis = 2, keepdims=True)
h2_var = h2.var(axis = 2, keepdims=True)
preln2 = (h2 - h2_mean)/((h2_var + 1e-5) ** 0.5)
ln2 = g2 * preln2 + b2

# ReLu
rl2 = np.maximum(np.zeros_like(ln2), ln2)

## Layer 3
# Final Linear Layer
h3 = rl2 @ W3 + B3
h3 = h3.squeeze(axis=-1)

# Softmax
logmax = np.max(h3, keepdims=True, axis=1)
logits = h3 - logmax
raw_probs = np.e ** logits
logits_sum = np.sum(raw_probs, axis = 1, keepdims=True)
probs = raw_probs * logits_sum ** -1

# Sparse Categorical Crossentropy
current_batch_size = xbatch.shape[0]
loss = -np.log(probs[np.arange(current_batch_size), ybatch])
print(f"Testing Loss: {loss.mean()}")

In [ ]:
### Sampling from the Neural Network
print("Type any sentence and the model will find the best word to cloze!")
sentence = input(">>> ")

## Tranforming The Sentences to Subword Embeddings
words = sentence.split()
enc_senti = list(map(tokenize_char, words))
# Adding Padding Words for the rest of the sentence
if len(enc_senti) < MAX_SENT_LEN:
  padding = [0] * MAX_WORD_LEN
  enc_senti.extend(padding for i in range(MAX_SENT_LEN - len(enc_senti)))
else:
  print(f"ERROR 3: The sentence length exceeded mpls by {len(enc_senti) - MAX_SENT_LEN}")

enc_senti = np.array(enc_senti)
enc_words = enc_senti.reshape(-1, MAX_WORD_LEN)
grams = lst.sliding_window_view(enc_words, window_shape=4, axis = 1).reshape(-1, 4) # All Four Grams
smpl_void = np.ascontiguousarray(grams).view(gramvoid).ravel()

indices = lookup(vocab_void, smpl_void, unk_idx)
indices = indices.reshape(1, MAX_SENT_LEN, -1)

### Running It Through The Model
## Batching
xbatch = indices

## Layer 0
# Embedding the subword tokens
emb = C[xbatch]
embcat = emb.reshape(xbatch.shape[0], xbatch.shape[1], -1)
embcat_shape = embcat.shape
embcat_dtype = embcat.dtype

## Layer 1
# Padding the sentences
padding_word = np.tile(C[0], 32)
pad_block = np.tile(padding_word, (embcat.shape[0], 2, 1))
pembcat = np.concatenate([pad_block, embcat, pad_block], axis=1)

# Applying the Slider
win1 = lst.sliding_window_view(pembcat, window_shape=WIN_SIZE, axis=1)
win1r = win1.reshape(win1.shape[:-2] + (-1,))
win1r = np.ascontiguousarray(win1r) # Faster for Matrix Mult

# First layer output
h1 = win1r @ W1 + B1

# LayerNorm
h1_mean = h1.mean(axis = 2, keepdims=True)
h1_var = h1.var(axis = 2, keepdims=True)
preln1 = (h1 - h1_mean)/((h1_var + 1e-5) ** 0.5)
ln1 = g1 * preln1 + b1

# ReLu 1
rl1 = np.maximum(np.zeros_like(ln1), ln1)

## Layer 2
# Padding the Array
rl1_padded = np.pad(rl1, ((0,0), (2,2), (0,0)), mode='constant', constant_values=0)

# Applying the Slider
win2 = lst.sliding_window_view(rl1_padded, window_shape = WIN_SIZE, axis = 1)
win2r = win2.reshape(win2.shape[:-2] + (-1,))
win2r = np.ascontiguousarray(win2r)

# Matrix Multiplying with Weights
h2 = win2r @ W2 + B2

# LayerNorm again
h2_mean = h2.mean(axis = 2, keepdims=True)
h2_var = h2.var(axis = 2, keepdims=True)
preln2 = (h2 - h2_mean)/((h2_var + 1e-5) ** 0.5)
ln2 = g2 * preln2 + b2

# ReLu
rl2 = np.maximum(np.zeros_like(ln2), ln2)

## Layer 3
# Final Linear Layer
h3 = rl2 @ W3 + B3
h3 = h3.squeeze(axis=-1)

# Softmax
logmax = np.max(h3, keepdims=True, axis=1)
logits = h3 - logmax
raw_probs = np.e ** logits
logits_sum = np.sum(raw_probs, axis = 1, keepdims=True)
probs = raw_probs * logits_sum ** -1
ans_index = np.argmax(probs)

print("Word Chosen: ", words[ans_index])

words[ans_index] = len(words[ans_index]) * '-'
print(' '.join(words))

In [ ]:
### NEURAL NETWORK DIAGNOSTIC WORK

In [ ]:
# Smoothing the loss for a cleaner visualization
window_size = 100
smoothed_loss = pd.Series(lossi).rolling(window=window_size).mean()

plt.figure(figsize=(10, 6))
plt.plot(steppi, lossi, alpha=0.2, label='Raw Loss')
plt.plot(steppi, smoothed_loss, color='red', label=f'Moving Avg ({window_size})')
# plt.axvline(x=max_steps*grad_descent, color='red', linestyle='--', label='Gradient Descent')
plt.yscale('log')
plt.xlabel('Step')
plt.ylabel('Loss (Log Scale)')
plt.title('Training Loss Convergence')
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Activation Histograms
plt.figure(figsize=(18, 5))
activations = [rl1, rl2, probs]
layer_names = ['Layer 1 (ReLU)', 'Layer 2 (ReLU)', 'Output (Softmax)']

for i, (act, name) in enumerate(zip(activations, layer_names)):
    plt.subplot(1, len(activations), i + 1)
    plt.hist(act.flatten(), bins=50, density=True, color='teal', alpha=0.7)

    if 'ReLU' in name:
        dead_pct = (act == 0).mean() * 100
        plt.title(f"{name}\nDead Neurons: {dead_pct:.1f}%")
    else:
        plt.title(f"{name}\nMean Prob: {act.mean():.4f}")
plt.tight_layout()
plt.show()

In [ ]:
# Gradient Histograms (Backward Pass Health)
plt.figure(figsize=(18, 5))
# Plotting the gradients of the weights to see how the learning signal propagates backward
weight_grads = [dW1, dW2, dW3]
grad_names = ['dW1', 'dW2', 'dW3']

for i, (grad, name) in enumerate(zip(weight_grads, grad_names)):
    plt.subplot(1, len(weight_grads), i + 1)
    plt.hist(grad.flatten(), bins=50, density=True, color='coral', alpha=0.7)
    plt.title(f"{name} Gradient\nSpread (std): {grad.std():.6f}")
plt.tight_layout()
plt.show()

In [ ]:
# Update-to-Data Ratio (Learning Dynamics)
plt.figure(figsize=(12, 6))

# Filter out the first step if it's empty (k=0 case in training loop)
if len(ud) > 0 and len(ud[0]) == 0:
    ud_plot = ud[1:]
    stepi_plot = steppi[1:]
else:
    ud_plot = ud
    stepi_plot = steppi

ud_array = np.array(ud_plot) # Shape: (steps, len(parameters))

# In your parameters list, W1 is index 0, W2 is index 3, and W3 is index 6
plt.plot(stepi_plot, ud_array[:, 1], label='W1', alpha=0.8)
plt.plot(stepi_plot, ud_array[:, 5], label='W2', alpha=0.8)
plt.plot(stepi_plot, ud_array[:, 9], label='W3', alpha=0.8)

# The golden target line
plt.axhline(-3, color='black', linestyle='--', label='Optimal Target ($10^{-3}$)')

plt.xlabel('Training Steps')
plt.ylabel('$log_{10}$(Update std / Parameter std)')
plt.title('Update-to-Data Ratio Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()